# Topic: LLM for Recommendation Systems
---

Version: Aug 2025 Written by Jin Yuze (jin.yuze@u.nus.edu)

###################

Under Construction

###################

---
## Introduction

In this topic, we want to explore the cutting edge of using Large Language Models (LLMs) for recommendation systems. LLMs have shown great potential in understanding and generating human-like text, and they can be leveraged to enhance the recommendation process by providing more personalized and context-aware suggestions.

In recent years, there has been a surge of interest in using LLMs for recommendation systems. These models can analyze user preferences, item descriptions, and contextual information to generate recommendations that are more aligned with user interests. The models can take the textual data associated with items (such as product descriptions, reviews, and user-generated content) and use it to better understand the relationships between users and items, leading to more accurate and relevant recommendations, compare to conventional collaborative filtering or content-based methods.

In this topic, we will explore this area following a reseach paper that proposes a noval approach.

The code is modified from project `LLM4REC`

[LLM-Enhanced User-Item Interactions: Leveraging Edge Information for Optimized Recommendations](https://arxiv.org/abs/2402.09617)

[Code Repo](https://github.com/anord-wang/LLM4REC)

#### Prelude

We assume you already have some basic knowledges on: 

- Conventional recommendation systems, such as collaborative filtering and content-based filtering, which are the already covered content in this course.
- Basic knowledge of Large Language Models (LLMs), such as Natural Language Processing (NLP) tasks, and how LLMs can be used to process and generate text, Attention Mechanism, Transformers, etc. (We have covered these basics in the course as well).
- Basic knowledge of PyTorch, such as how to define a model, how to train a model, and how to use PyTorch's built-in functions for optimization and loss calculation. 

#### Example Code
Since the whole code repo is quite complex, we will not go through all the components in this notebook. Instead, here we only show the main learning points of the topic. 

We provide a full version of the code repo as additional materials, which you are recommanded to try out by yourself.
We have provided a detailed instruction on how to run the code, and we provide the checkpoints for you to test the code without training the model from scratch (which will take some GPU resouces and long time).
And in the code repo, we provide detailed comments on each component, and a more technical explanation of the model architecture and training process.

---
## Overview

It is not trivial to use LLMs for recommendation systems, as the two systems have different assumptions on data structure. 
The paper we are going to explore proposes a novel approach that tries to fill in the gap between graph-based user & item interactions and LLMs' text-based understanding.

In previous studies in our module, we have seen the conventional recommendation methods, such as collaborative filtering and content-based filtering, which are based on user-item interactions. 
These methods typically rely on the assumption that user-item interactions can be represented as a bipartite graph, where users and items are nodes, and interactions (such as ratings or clicks) are edges. 
We have also seen how graph neural networks (GNNs) can be used to model these interactions, allowing for the aggregation of information from neighboring nodes to improve recommendations. 
These methods are shown to be effective in capturing the relationships between users and items.

While you have seen how LLMs can be used for various NLP tasks, such as text generation, sentiment analysis, and question answering, they are typically trained on large text corpora and are not directly designed to handle graph-based data. 
LLMs excel at understanding and generating text, but they do not inherently understand the structure of user-item interactions as a graph.

Thus how to bridge the gap between these two paradigms is a key challenge in using LLMs for recommendation systems.

---
### Idea

In this example code, we will explore one solution to this challenge.

The idea of this example paper is to leverage the edge information in user-item interactions to enhance the recommendations generated by LLMs.
The authors propose a novel approach that combines the strengths of both graph-based methods and LLMs, allowing for more effective recommendations:

- **Graph-aware Attention Mechanism** – The authors modify the LLM’s attention module by incorporating additional bias terms derived from the graph structure of the user-item interactions. This includes both direct connections (user-item links) and indirect relationships (shortest paths in the interaction graph). By integrating these structural signals into the attention computation, the model can focus more on semantically or structurally related nodes, rather than relying purely on sequential token relationships.

- **Tokenizer and Embedding Modifications** – To handle user and item IDs without losing their semantic identity, the authors adjust the tokenizer so that these IDs are treated as indivisible tokens. Special embeddings are then assigned to these tokens, enabling the LLM to recognize and represent specific users and items consistently during training and inference.

- **Prompt Design** – The input to the LLM is carefully constructed to include textual descriptions, user profiles, and the graph-based interaction information. This design ensures the model receives both semantic content and structural context, improving its ability to generate relevant recommendations.

- **Two-stage Training Strategy** – The training process consists of a pretraining stage, where the model learns general patterns from the full user-item corpus using a text generation objective, and a fine-tuning stage, where the model is optimized for the recommendation task by predicting the most likely items for a given user prompt.

### Terminology and Problem Statement:

In our description, we will follow the paper's terminology and problem statement, which we copy from the paper: 

> Consider the existence of `I` users and `J` items, let `X`<sub>`ij`</sub> be the binary interaction (e.g., purchase) matrix between user `i` and item `j`. 
> Besides, we collect user descriptions, item descriptions (e.g., prices, brand, category, title), user reviews for items, and explanations of user purchase reasons.
> We denote `T`<sub>`i`</sub> as the descriptions of the user `i`, `T`<sub>`j`</sub> as the descriptions of the item `j`, `T`<sub>`ij`</sub> as the joint texts of the user `i` and item `j`, such as user reviews and purchase reasons for items. 
> We unify all textual descriptions into `T` that includes `N` sequences, `k` indexes the tokens in each sequence, and `T`<sub>`nk`</sub> is the `𝑘`-th token in the `n`-th sequence. 
> Our goal is to leverage LLMs and graphs to develop a generative recommender system that takes a prompt, including a user ID and a user’s historical interaction records with items, and generates product recommendations to the user.

---

### Step-1 Attention may not be enough

Let's start from looking at our data. 

In our data, we have the following information:
- User's profile, Item's description, User's reviews. These are textual data. LLM can learn from these data to understand the user's preferences and item characteristics.
- User-item interactions, and second-order relationships between items. These are graph-based data. We can use these data to enhance the LLM's understanding of the relationships between users and items.

For the textual data, it is straightforward to use LLMs to process them. We can simply feed the text into the LLM and let it generate embeddings or predictions based on the text.

For the graph-based data, we need to do some extra work to enable the LLM to process them.

Previous, Attention mechanism is the key component to capture the relationships between tokens in the **text**. However, when we provide information like: 
```
<user_123> has interaction with <item_abc>
```
The attention mechanism can only provide the "language" relationship between the tokens `<user_123>` and `<item_abc>`, while the actual graph-based relationship is missing, thus the model may not fully understand the relationship between the user and the item.

To address this, we need to modify the attention mechanism to incorporate the graph-based relationships.

The original attention is defined as: 
<math display="block">
<mi>A</mi><mi>t</mi><mi>t</mi><mo stretchy="false">(</mo><mi>Q</mi><mo>,</mo><mi>K</mi><mo>,</mo><mi>V</mi><mo stretchy="false">)</mo><mo>=</mo><mi>S</mi><mi>o</mi><mi>f</mi><mi>t</mi><mi>m</mi><mi>a</mi><mi>x</mi><mo stretchy="false">(</mo><mfrac><mrow><mi>Q</mi><msup><mi>K</mi><mi>T</mi></msup></mrow><msqrt><msub><mi>d</mi><mi>k</mi></msub></msqrt></mfrac><mo stretchy="false">)</mo><mi>V</mi>
</math>
(We will omit the detailed explanation of the attention mechanism here.)

As you can see, the attention is only defined based on the textural relationship between the tokens, thus, in the example paper, the Attention between `user` or `item` is modified to: 
<math display="block">
<mi>A</mi><mi>t</mi><msup><mi>t</mi><mo>′</mo></msup><mo stretchy="false">(</mo><mi>Q</mi><mo>,</mo><mi>K</mi><mo>,</mo><mi>V</mi><mo stretchy="false">)</mo><mo>=</mo><mi>S</mi><mi>o</mi><mi>f</mi><mi>t</mi><mi>m</mi><mi>a</mi><mi>x</mi><mo stretchy="false">(</mo><mfrac><mrow><mi>Q</mi><msup><mi>K</mi><mi>T</mi></msup></mrow><msqrt><msub><mi>d</mi><mi>k</mi></msub></msqrt></mfrac><mo>+</mo><mi>R</mi><mo stretchy="false">)</mo><mi>V</mi>
</math>

with an additional bias term `R` that is defined as: 
<math display="block">
<mi>R</mi><mo>=</mo><msup><mi>R</mi><mrow><mi>c</mi><mi>o</mi><mi>n</mi><mi>n</mi></mrow></msup><mo>+</mo><msup><mi>R</mi><mrow><mi>p</mi><mi>a</mi><mi>t</mi><mi>h</mi></mrow></msup>
</math>
where `R`<sub>`con`</sub> is the direct connection between the user and item, and `R`<sub>`path`</sub> is the second-order relationship between items.

For the direct connection, it is easily defined as: 
<math display="block"><msubsup><mi>R</mi><mrow><mi>i</mi><mi>j</mi></mrow><mrow><mrow><mi mathvariant="normal">c</mi><mi mathvariant="normal">o</mi><mi mathvariant="normal">n</mi><mi mathvariant="normal">n</mi></mrow></mrow></msubsup><mo>=</mo><mrow data-mjx-texclass="INNER"><mo data-mjx-texclass="OPEN">{</mo><mtable columnalign="left left" columnspacing="1em" rowspacing=".2em"><mtr><mtd><mn>1</mn><mo>,</mo></mtd><mtd><mrow><mtext>if there is a direct connection between node&nbsp;</mtext><mrow><mi>i</mi></mrow><mtext>&nbsp;and&nbsp;</mtext><mrow><mi>j</mi></mrow><mtext>,</mtext></mrow></mtd></mtr><mtr><mtd><mn>0</mn><mo>,</mo></mtd><mtd><mtext>otherwise.</mtext></mtd></mtr></mtable><mo data-mjx-texclass="CLOSE" fence="true" stretchy="true" symmetric="true"></mo></mrow></math>.

This represents the strong direct interaction between the user and the item.

While for the indirect connection, it is corresponding to the "collaborative information" in conventional solutions. It is defined as:
```
A normalized shortest path score between nodes, which is computed based on the entire graph.
```
The formula is: 
<math display="block"><msubsup><mi>R</mi><mrow><mi>i</mi><mi>j</mi></mrow><mrow><mi>p</mi><mi>a</mi><mi>t</mi><mi>h</mi></mrow></msubsup><mo>=</mo><mn>1</mn><mo>−</mo><mfrac><mrow><mi>δ</mi><msub><mi>P</mi><mrow><mi>i</mi><mi>j</mi></mrow></msub></mrow><mrow><mi>m</mi><mi>a</mi><mi>x</mi><mo stretchy="false">(</mo><mi>P</mi><mo stretchy="false">)</mo></mrow></mfrac></math>
Where `P` is the length of the shortest path between nodes `i` and `j`. and `δ` is a factor between 0 and 1. 

With this modified Attention mechanism, now the model can capture both the textual relationships and the graph-based relationships between users and items.

TODO: Example code

---

### Step-2 Take care of the Tokenizer

But for the graph-based data, we need to do some extra work to make it compatible with LLMs. There is a key challenge: The user-ID and item-ID are actually not typical text, means when you receive an ID like `user_123`, it should be processed as a whole token, rather than being split into sub-tokens like `user`, `_`, and `123`. This is because the ID is a unique identifier, and splitting it into sub-tokens would lose its meaning. This requires us to modify the tokenization process of the LLM to handle these IDs correctly.


---

### Step-3 Prompt Design

Before we start training the model, we need to prepare the input training data for the LLM. 

This requires us to design a set of prompts as a structured textual data. 



---

### Step-4 Pre-training and Fine-tuning
With the modifications, we can now perform a pre-training step to train the LLM on our data.

The training is usually performed in two stages: pre-training and fine-tuning.

The purpose of this pre-training step is to let the LLM learn the general structure of the data, including the relationships between users and items, and how to generate text based on these relationships.

After the pre-training, we would need to fine-tune the LLM to be more "good at" the recommendation tasks. 

We generate the input data based on each user's historical interactions. Then we fine-tune the LLM on this data, to optimize it to minimize recommendation errors, not textural generation likelihood. 